In [1]:
import sqlite3
import numpy as np
import pandas as pd
import json
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import requests
import time
import re

In [2]:
file = "database.db"

try:
  conn = sqlite3.connect(file)
  print(f"Database {file} formed.")
except:
  print(f"Database {file} not formed.")

Database database.db formed.


In [3]:
try:
    sqliteConnection = sqlite3.connect(file)
    print("Connected to SQLite")
except sqlite3.Error as error:
    print("Failed to connect with sqlite3 database", error)

Connected to SQLite


In [4]:
cursor = sqliteConnection.cursor()

In [5]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS universities (
        id INTEGER AUTO_INCREMENT PRIMARY KEY,
        name TEXT NOT NULL,
        base_fee NUMBER,
        type_fee TEXT
    )
''')

In [6]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS bachelors (
        id INTEGER AUTO_INCREMENT PRIMARY KEY,
        name TEXT NOT NULL,
        university_id NUMBER,
        spe_fee NUMBER,
        emp_rate NUMBER,
        FOREIGN KEY (university_id) REFERENCES universities(id)
    )
''')

In [7]:
df = pd.DataFrame(columns=["name","university","spe_fee","emp_rate"])

In [15]:
universities_l = ["UNAM","TEC MONTERREY"]

unam_careers = [
    # ÁREA 1
    "Ingeniería Civil",
    "Ingeniería en Computación",
    "Ingeniería Industrial",
    "Ingeniería Eléctrica",
    "Ingeniería Mecánica",

    # ÁREA 2
    "Médico Cirujano",
    "Psicología",
    "Biología",
    "Ingeniería Química",

    # ÁREA 3
    "Derecho",
    "Administración",
    "Economía",
    "Comunicación",

    # ÁREA 4
    "Arquitectura",
    "Diseño Gráfico",
    "Urbanismo",
    "Historia",
]

tec_careers = [
    # ÁREA 1
    "Ingeniería Civil",
    "Ingeniería en Tecnologías Computacionales",
    "Ingeniería Industrial y de Sistemas",
    "Ingeniería en Electrónica y Semiconductores",
    "Ingeniería Mecánica",

    # ÁREA 2
    "Médico Cirujano",
    "Licenciatura en Psicología Clínica y de la Salud",
    "Licenciatura en Biociencias",
    "Ingeniería Química",

    # ÁREA 3
    "Licenciatura en Derecho",
    "Licenciatura en Estrategia y Transformación de Negocios",
    "Licenciatura en Finanzas",
    "Licenciatura en Comunicación y Producción Digital",

    # ÁREA 4
    "Licenciatura en Arquitectura",
    "Licenciatura en Diseño",
    "Licenciatura en Urbanismo",
    "Licenciatura en Humanidades Digitales e Inteligencia Artificial",
]
qs_links = {
    # ÁREA 1 – Engineering and Technology
    "Ingeniería Civil": "https://www.topuniversities.com/university-subject-rankings/civil-structural-engineering",
    "Ingeniería en Computación": "https://www.topuniversities.com/university-subject-rankings/computer-science-information-systems",
    "Ingeniería Industrial": "https://www.topuniversities.com/university-subject-rankings/mechanical-aeronautical-manufacturing-engineering",
    "Ingeniería Eléctrica": "https://www.topuniversities.com/university-subject-rankings/electrical-electronic-engineering",
    "Ingeniería Mecánica": "https://www.topuniversities.com/university-subject-rankings/mechanical-aeronautical-manufacturing-engineering",

    # ÁREA 2 – Life Sciences & Medicine
    "Médico Cirujano": "https://www.topuniversities.com/university-subject-rankings/medicine",
    "Psicología": "https://www.topuniversities.com/university-subject-rankings/psychology",
    "Biología": "https://www.topuniversities.com/university-subject-rankings/biological-sciences",
    "Ingeniería Química": "https://www.topuniversities.com/university-subject-rankings/chemical-engineering",

    # ÁREA 3 – Social Sciences & Management
    "Derecho": "https://www.topuniversities.com/university-subject-rankings/law-legal-studies",
    "Administración": "https://www.topuniversities.com/university-subject-rankings/business-management-studies",
    "Economía / Finanzas": "https://www.topuniversities.com/university-subject-rankings/economics-econometrics",
    "Comunicación": "https://www.topuniversities.com/university-subject-rankings/communication-media-studies",

    # ÁREA 4 – Arts & Humanities
    "Arquitectura": "https://www.topuniversities.com/university-subject-rankings/architecture-built-environment",
    "Diseño Gráfico": "https://www.topuniversities.com/university-subject-rankings/art-design",
    "Urbanismo": "https://www.topuniversities.com/university-subject-rankings/architecture-built-environment",
    "Historia": "https://www.topuniversities.com/university-subject-rankings/art-history"
}


In [16]:
df.head()

,name,university,spe_fee,emp_rate


In [17]:
def getEmployer_Reputation(university,id,array_careers,qs_links):
    filas = []
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Ejecutar en segundo plano
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(options=chrome_options)
    try:
        for i in range(len(array_careers)):
            prepath = list(qs_links.values())[i]
            url = prepath + f"?search={university}"
            print(url)
            driver.get(url)
            time.sleep(5)
            page_source = driver.page_source
            if university in page_source.lower():
                elemento = driver.find_element(By.CLASS_NAME, "new-rankings-ind-val")
                emp_rate = elemento.text
                print(f"Emp rate {emp_rate}")
            else:
                emp_rate = None
                print("No se encontró el bloque de Employer Reputation.")
            filas.append({"name":array_careers[i], "university":id, "spe_fee":None, "emp_rate":emp_rate})
    finally:
        driver.quit()
        print("Completado")
    return filas

In [18]:
unam_filas = getEmployer_Reputation("unam",1,unam_careers,qs_links)

https://www.topuniversities.com/university-subject-rankings/civil-structural-engineering?search=unam
Emp rate 81.7
https://www.topuniversities.com/university-subject-rankings/computer-science-information-systems?search=unam
Emp rate 83
https://www.topuniversities.com/university-subject-rankings/mechanical-aeronautical-manufacturing-engineering?search=unam
Emp rate 80
https://www.topuniversities.com/university-subject-rankings/electrical-electronic-engineering?search=unam
Emp rate 74.5
https://www.topuniversities.com/university-subject-rankings/mechanical-aeronautical-manufacturing-engineering?search=unam
Emp rate 80
https://www.topuniversities.com/university-subject-rankings/medicine?search=unam
Emp rate 78.6
https://www.topuniversities.com/university-subject-rankings/psychology?search=unam
Emp rate 88.1
https://www.topuniversities.com/university-subject-rankings/biological-sciences?search=unam
Emp rate 83.6
https://www.topuniversities.com/university-subject-rankings/chemical-engineeri

In [20]:
tec_filas = getEmployer_Reputation("monterrey",2,tec_careers,qs_links)

https://www.topuniversities.com/university-subject-rankings/civil-structural-engineering?search=monterrey
Emp rate 82.8
https://www.topuniversities.com/university-subject-rankings/computer-science-information-systems?search=monterrey
Emp rate 85.3
https://www.topuniversities.com/university-subject-rankings/mechanical-aeronautical-manufacturing-engineering?search=monterrey
Emp rate 81
https://www.topuniversities.com/university-subject-rankings/electrical-electronic-engineering?search=monterrey
Emp rate 77.8
https://www.topuniversities.com/university-subject-rankings/mechanical-aeronautical-manufacturing-engineering?search=monterrey
Emp rate 81
https://www.topuniversities.com/university-subject-rankings/medicine?search=monterrey
Emp rate 73.2
https://www.topuniversities.com/university-subject-rankings/psychology?search=monterrey
Emp rate 89.9
https://www.topuniversities.com/university-subject-rankings/biological-sciences?search=monterrey
Emp rate 76.5
https://www.topuniversities.com/univ

In [21]:
unam_filas

[{'name': 'Ingeniería Civil',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '81.7'},
 {'name': 'Ingeniería en Computación',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '83'},
 {'name': 'Ingeniería Industrial',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '80'},
 {'name': 'Ingeniería Eléctrica',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '74.5'},
 {'name': 'Ingeniería Mecánica',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '80'},
 {'name': 'Médico Cirujano',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '78.6'},
 {'name': 'Psicología', 'university': 1, 'spe_fee': None, 'emp_rate': '88.1'},
 {'name': 'Biología', 'university': 1, 'spe_fee': None, 'emp_rate': '83.6'},
 {'name': 'Ingeniería Química',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '80.5'},
 {'name': 'Derecho', 'university': 1, 'spe_fee': None, 'emp_rate': '84.2'},
 {'name': 'Administración',
  'university': 1,
  'spe_fee': None,
  'emp_rate': '82.1'},
 {'name': 'Economía', 'uni

In [22]:
tec_filas

[{'name': 'Ingeniería Civil',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '82.8'},
 {'name': 'Ingeniería en Tecnologías Computacionales',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '85.3'},
 {'name': 'Ingeniería Industrial y de Sistemas',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '81'},
 {'name': 'Ingeniería en Electrónica y Semiconductores',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '77.8'},
 {'name': 'Ingeniería Mecánica',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '81'},
 {'name': 'Médico Cirujano',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '73.2'},
 {'name': 'Licenciatura en Psicología Clínica y de la Salud',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '89.9'},
 {'name': 'Licenciatura en Biociencias',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '76.5'},
 {'name': 'Ingeniería Química',
  'university': 2,
  'spe_fee': None,
  'emp_rate': '75'},
 {'name': 'Licenciatura en Derecho',
  'university': 2,
  'spe_fee': 

In [23]:
df = pd.concat([df, pd.DataFrame(unam_filas, columns=df.columns)], ignore_index=True)
df = pd.concat([df, pd.DataFrame(tec_filas, columns=df.columns)], ignore_index=True)

In [33]:
area_1 = [
    "ciencias_computacionales",
    "construcción_e_ingeniería_civil",
    "electricidad_y_generación_de_energía",
    "electrónica_automatización_y_aplicaciones_de_la_mecánica_eléctrica",
    "mecánica_y_profesiones_afines_al_trabajo_metálico"
]

area_2 = [
    "biología",
    "ingeniería_de_procesos_químicos",
    "medicina_general",
    "psicología"
]

area_3 = [
    "administración_de_empresas",
    "derecho",
    "economía",
    "comunicación_y_periodismo"
]

area_4 = [
    "arquitectura_y_urbanismo",
    "diseño_y_comunicación_gráfica_y_editorial",
    "historia_y_arqueología",
    "arquitectura_y_urbanismo"
]


In [ ]:
carreras